In [1]:
import sys

In [2]:
sys.path.insert(0, "..")

In [3]:
import spacy
from spacy.tokens import Span

import medspacy
from medspacy.preprocess import PreprocessingRule, Preprocessor
from medspacy.ner import TargetRule
from medspacy.context import ConTextRule
from medspacy.section_detection import Sectionizer
from medspacy.postprocess import PostprocessingRule, PostprocessingPattern, Postprocessor
from medspacy.postprocess import postprocessing_functions
from medspacy.visualization import visualize_ent, visualize_dep


import re

# Overview
In this notebook, we'll show how to use a pretrained model for target concept extraction instead of defining rules. We'll then add our additional components to show how medSpaCy can be used to combine statistical NLP with other rule-based components.

As an example, we'll download the [med7](https://github.com/kormilitzin/med7) transformers model which can be used with spacy 3. This won't get all the concepts we're interested in, but will extract drug-related information like names and doses.

We can install this model with `pip` using this GitHub link:
```bash
pip install --no-deps https://huggingface.co/kormilitzin/en_core_med7_trf/resolve/main/en_core_med7_trf-any-py3-none-any.whl
```

**NOTE:** As of, Oct 2022, med7 supports spacy 3.1 only. Because medspaCy now supports spacy 3.3 or higher, please install the model with no deps to keep the spacy version in the environment the same. The model still functions as of 3.4.1.

We'll also need to install `spacy-transformers`:
```bash
pip install spacy-transformers
```

In [4]:
# !pip install --no-deps https://huggingface.co/kormilitzin/en_core_med7_trf/resolve/main/en_core_med7_trf-any-py3-none-any.whl

In [5]:
# !pip install spacy-transformers

In [6]:
with open("./discharge_summary.txt") as f:
    text = f.read()

This model now can be loaded as any other spaCy model. We'll use `medspacy.load()` and pass in this model name.

In [8]:
#nlp = medspacy.load("en_core_med7_trf")
nlp = medspacy.load("en_core_med7_lg")
#nlp = medspacy.load("en_core_web_trf")

In [9]:
nlp.pipe_names

['tok2vec',
 'ner',
 'medspacy_pyrush',
 'medspacy_target_matcher',
 'medspacy_context']

In [10]:
ner = nlp.get_pipe("ner")

In [11]:
ner.labels

('DOSAGE', 'DRUG', 'DURATION', 'FORM', 'FREQUENCY', 'ROUTE', 'STRENGTH')

In [12]:
doc = nlp(text)

In [13]:
doc.ents

(Hydrochlorothiazide,
 chemo,
 chemo,
 Miconazole Nitrate,
 2 %,
 Powder,
 One (1),
 Appl,
 Topical,
 2 times a day,
 Solution,
 One (1),
 Injection,
 TID (3 times a day),
 Acetaminophen,
 160 mg/5 mL,
 Elixir,
 One (1),
 PO,
 Q4-6H)

## Process our text
Similar to the last notebook, we'll add new rules to some of our components. Let's first look at what our model extracts out of the box:

In [14]:
visualize_ent(doc)

### Preprocessing

In [15]:
preprocessor = Preprocessor(nlp.tokenizer)

In [16]:
nlp.tokenizer = preprocessor

In [17]:
preprocess_rules = [
    
    PreprocessingRule(
        r"\[\*\*[\d]{1,4}-[\d]{1,2}(-[\d]{1,2})?\*\*\]",
        repl="01-01-2010",
        desc="Replace MIMIC date brackets with a generic date."
    ),
    
    PreprocessingRule(
        r"\[\*\*[\d]{4}\*\*\]",
        repl="2010",
        desc="Replace MIMIC year brackets with a generic year."
    ),
    
    PreprocessingRule(
        r"dx'd", 
        repl="Diagnosed", 
        desc="Replace abbreviation"
    ),
    
    PreprocessingRule(
        r"tx'd", 
        repl="Treated", 
        desc="Replace abbreviation"
    ),
    
    PreprocessingRule(
        r"\[\*\*[^\]]+\]", 
        desc="Remove all other bracketed placeholder text from MIMIC"
    )
]

In [18]:
preprocessor.add(preprocess_rules)

### Target Rules
The trained NER will add some new concepts that we weren't getting before, but we can customize with rules.

In [19]:
target_rules = [
    TargetRule(literal="abdominal pain", category="PROBLEM"),
    TargetRule("stroke", "PROBLEM"),
    TargetRule("hemicolectomy", "TREATMENT"),
    TargetRule("colon cancer", "PROBLEM"),
    TargetRule("radiotherapy", "PROBLEM",
              pattern=[{"LOWER": "xrt"}]),
    TargetRule("metastasis", "PROBLEM"),
    
    TargetRule("Type II Diabetes Mellitus", "PROBLEM", 
              pattern=[
                  {"LOWER": "type"},
                  {"LOWER": {"IN": ["2", "ii", "two"]}},
                  {"LOWER": {"IN": ["dm", "diabetes"]}},
                  {"LOWER": "mellitus", "OP": "?"}
              ],),
    TargetRule("Hypertension", "PROBLEM",
              pattern=[{"LOWER": {"IN": ["htn", "hypertension"]}}],),
    
]

In [20]:
target_matcher = nlp.get_pipe("medspacy_target_matcher")

In [21]:
target_matcher.add(target_rules)

### Context

In [22]:
context = nlp.get_pipe("medspacy_context")

In [23]:
context_rules = [
    ConTextRule("diagnosed in <YEAR>", "HISTORICAL", 
               pattern=[
                   {"LOWER": "diagnosed"},
                   {"LOWER": "in"},
                   {"LOWER": {"REGEX": "^[\d]{4}$"}}
               ])
]

In [24]:
context.add(context_rules)

### Section detection

In [25]:
sectionizer = nlp.add_pipe("medspacy_sectionizer")

In [26]:
from medspacy.section_detection import SectionRule

In [27]:
section_rule = SectionRule("Brief Hospital Course:", "hospital-course")

In [28]:
sectionizer.add(section_rule)

### Postprocessing
Here, we'll show another example of how postprocessing can be used. The NER component extracts **"married"** as a **"TREATMENT"** entity. While some might agree with this in a philosophical sense, it doesn't match our clinical definition very well. This shows a challenge of statistical NLP: we have relatively little control over what concepts are extracted by our model. But we can use some postprocessing rules to clean this up.

Postprocessing can be used to remove or clean up entities which we know are incorrect. In this example, we'll just remove any entity where the text is **"married"**:

In [29]:
postprocessor = nlp.add_pipe("medspacy_postprocessor")

In [30]:
postprocess_rules = [
    PostprocessingRule(
        patterns=[
            PostprocessingPattern(condition=lambda ent: ent.text.lower(), success_value="married"),
        ],
        action=postprocessing_functions.remove_ent,
        description="Remove a specific misclassified span of text."
    ),
    
]

In [31]:
postprocessor.add(postprocess_rules)

# Process our document
Now, let's process the text with our complete pipeline and show the results:

In [32]:
nlp.pipe_names

['tok2vec',
 'ner',
 'medspacy_pyrush',
 'medspacy_target_matcher',
 'medspacy_context',
 'medspacy_sectionizer',
 'medspacy_postprocessor']

In [33]:
doc = nlp(text)

In [34]:
visualize_ent(doc)

<hr style="height:0.25rem;">

Example using shorter text

In [35]:
short_text = "Colon cancer dx'd in [**2554**], tx'd with hemicolectomy, chemo"
short_doc = nlp(short_text)

In [36]:
visualize_ent(short_doc)

In [37]:
visualize_dep(short_doc)

<hr style="height:0.5rem;">

### CodiEsp (Article ID: S1131-57682001000100007-2)

- ICD Codes: F19.20 I10 B19.20 K76.6 L98.9 T14.91 K74.60 F11.20 R16.1
- Text References: intravenous drug addiction; hypertension; HCV; portal hypertension; skin lesions; suicide; cirrhosis; methadone replacement therapy; splenomegaly

In [38]:
short_text = """
A 50-year-old male weighing 79 kg with HCV cirrhosis (type 1b/scan 22.6 KPa) diagnosed 20 years ago with fibrotic stenosis, portal hypertension and leukothrombocytis.
Background of parenteral methadone maintenance therapy.
In 2009 treatment with non-effective interferon.
Return of medical and family history without relevance.
In July 2015 she started treatment with ViekiraxÂ®, ExactÂ® and RebetolÂ®, 1200 mg a day for 12 weeks, without any concomitant treatment.
Two weeks later she went to the doctor's office referring good tolerance.
At week 4 the viral load is already negative but the patient reports skin lesions, change of character, aggressiveness and ideas of suicide, so both the patient and the family request to suspend treatment and is referred to Psychiatry.
In December 2015, the patient came to consultation with a Psychiatric Report in which it was stated that no psychotropic drug was prescribed and that at that time there was no contraindication to start another treatment for HCV.
In February 2016, treatment was initiated with ledipasvir 90 mg/soterovir 400 mg (rvoniÂ®) and RebetolÂ® 1,200 mg for 12 weeks.
At week 4 the patient reported good tolerance and the viral load was negative.
At week 12, the patient successfully completed treatment.
"""
short_doc = nlp(short_text)
visualize_ent(short_doc)

<hr style="height:0.25rem;">

Using the Text Reference

In [39]:
short_text = """
intravenous drug addiction; hypertension; HCV; portal hypertension; skin lesions; suicide; cirrhosis; methadone replacement therapy; splenomegaly
"""
short_doc = nlp(short_text)
visualize_ent(short_doc)

<hr style="height:0.25rem;">

Using the ICD Code Titles

<hr style="height:0.5rem;">

### CodiEsp (Article ID: S1130-14732006000200011-1)

- ICD Codes: C79.51 M89.70 N94.89 S06.9X B030 BW25 BN20 I61.8 0NB0 Q75.0 D66 S06.36 00C0 D49.2 BN00ZZZ
- Text References: bone metastasis; bone defect; pelvis bruise; brain trauma; brain MRI; CT of chest, abdomen and pelvis; CT scan of skull; intraparenchymal brain hematoma; craniectomy; scaphocephaly; hemophilia A; brain bruise; brain removed hematoma; bone tumor; plain x-ray skull

In [40]:
short_text = """
A 21-year-old male was admitted to the Neurosurgery Department because of nodules in the left parietal region that have increased in volume in the last 7 months.
Relevant personal history included mild ataxia and surgical intervention for this condition in childhood (craniectomy + silicon bands + bone fixation with hemoclips).
Spontaneous intraparenchymal cerebral hematoma was treated conservatively.
There are no known trauma episodes.
Physical examination revealed a painless, non-pulsatile, surface-covered, elastic tumor with a normal scalp size of 5x3x3 cm in the left parietal region.
Neurological examination was normal.
1.
Laboratory findings were: hemoglobin, 16g/dl; hematocrit, 50%; leukocytes, 6710, with a normal differential count; platelets, 211000; prothrombin activity, 97%; APTT: 59sec.
Factor VIII: 19%.
Plain radiography and skull CT showed an osteolytic lesion of sclerotic margins at the left parietal level with erosion of the external and internal table and expansion of the diploe.
MRI revealed a well-defined parietal nodule on the same side, with different signal areas inside, with hemoglobin at different stages of evolution: high signal density for methaemoglobin and low signal density for hemoglobin.
1.
A CT scan of the chest, abdomen and pelvis, as well as bone scan, ruled out bone tumor or metastasis.
The patient received 50U/kg/d Factor VIII prior to surgery, which increased her factor VIII level to 89% of normal.
A continuous intravenous infusion of 5U/kg/h of Factor VIII was administered throughout the operation.
1.
Surgical intervention
Under local anesthesia, a scalp incision was made over the tumor and a subperistal dissection was performed in different stages. An elevation and erosion of the external table was also observed, containing the diploe surrounding fibrous tissue.
The internal table was thinned, with irregular ballooning and bone defect.
Hematoma, granular tissue and foreign body (silicone) were removed, as well as the thinned and elevated external table, until reaching thick bone and normal appearance by means of bone curettage.
Histological examination confirmed the presence of old hematoma and granulation tissue.
After surgery, Factor VIII was maintained in 80% of normal activity until suture material was removed.
"""
short_doc = nlp(short_text)
visualize_ent(short_doc)

<hr style="height:0.25rem;">

Using the Text Reference

In [41]:
short_text = """
bone metastasis; bone defect; pelvis bruise; brain trauma; brain MRI; CT of chest, abdomen and pelvis; CT scan of skull; intraparenchymal brain
"""
short_doc = nlp(short_text)
visualize_ent(short_doc)

<hr style="height:0.25rem;">

Using the ICD Code Titles